# Import

In [4]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Load Data

In [5]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Drop ID column
df.drop(columns=["customerID"], inplace=True)

# Fix TotalCharges issue
df["TotalCharges"] = df["TotalCharges"].replace(" ", "0").astype(float)

# Encode target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# FEATURE ENGINEERING

In [6]:
df["AvgCharges"] = df["TotalCharges"] / (df["tenure"] + 1)

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[0, 12, 24, 48, 60, 72],
    labels=False
)

df["IsLongTerm"] = (df["tenure"] > 24).astype(int)

df["HighMonthlyCharges"] = (
    df["MonthlyCharges"] > df["MonthlyCharges"].median()
).astype(int)

# HANDLE MISSING VALUES

In [7]:
df = df.fillna(df.median(numeric_only=True))

# LABEL ENCODING

In [8]:
encoders = {}

for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# save encoders
with open("encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

# SPLIT DATA

In [10]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# SCALING (ONLY FOR LOGISTIC REGRESSION)

In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# MODELS

In [13]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [14]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=300,
                       random_state=42)

In [15]:
# XGBoost
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=3,
    random_state=42,
    eval_metric="logloss"
)
xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

# EVALUATION FUNCTION

In [16]:
def evaluate(name, model, X_t, scaled=False):
    if scaled:
        X_input = X_test_scaled
    else:
        X_input = X_t

    y_pred = model.predict(X_input)
    y_prob = model.predict_proba(X_input)[:, 1]

    print(f"\n🔹 {name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))

# RESULTS

In [17]:
evaluate("Logistic Regression", lr, X_test, scaled=True)
evaluate("Random Forest", rf, X_test)
evaluate("XGBoost", xgb, X_test)


🔹 Logistic Regression
Accuracy: 0.7416607523066004
ROC-AUC: 0.84502828799504
Confusion Matrix:
 [[752 283]
 [ 81 293]]
Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.75      0.71      1409
weighted avg       0.80      0.74      0.76      1409


🔹 Random Forest
Accuracy: 0.7743080198722498
ROC-AUC: 0.8401870366064739
Confusion Matrix:
 [[831 204]
 [114 260]]
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.80      0.84      1035
           1       0.56      0.70      0.62       374

    accuracy                           0.77      1409
   macro avg       0.72      0.75      0.73      1409
weighted avg       0.79      0.77      0.78      1409


🔹 XGBoost
Accuracy: 0.7494677075940384
ROC-AUC: 0.84078250536

# FEATURE IMPORTANCE (RF)

In [18]:
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop Features:\n", feat_imp.sort_values(ascending=False).head(10))


Top Features:
 Contract           0.145920
tenure             0.111883
MonthlyCharges     0.103085
TotalCharges       0.095554
AvgCharges         0.094364
OnlineSecurity     0.073312
TechSupport        0.056191
TenureGroup        0.053188
InternetService    0.037072
PaymentMethod      0.036011
dtype: float64
